In [ ]:
import os
import shutil
import subprocess
import tempfile
import threading
from pathlib import Path
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

class PDFPPTMerger:
    def __init__(self, root):
        self.root = root
        self.root.title("Document Merger")
        self.root.geometry("700x550")
        self.files = []

        self._setup_styles()
        self._build_ui()

    def _setup_styles(self):
        style = ttk.Style()
        style.theme_use('clam') # 'clam' looks cleaner on Windows/Linux than 'alt'
        style.configure("TButton", padding=6, font=('Segoe UI', 10))
        style.configure("Accent.TButton", foreground="white", background="#2196F3")

    def _build_ui(self):
        # Header
        header = ttk.Label(self.root, text="🗂 PDF & PPTX Merger", font=('Segoe UI', 16, 'bold'))
        header.pack(pady=10)

        # File Listbox
        self.frame_list = ttk.Frame(self.root)
        self.frame_list.pack(fill="both", expand=True, padx=20)

        self.listbox = tk.Listbox(self.frame_list, selectmode=tk.SINGLE, font=('Segoe UI', 10))
        self.listbox.pack(side="left", fill="both", expand=True)

        scrollbar = ttk.Scrollbar(self.frame_list, orient="vertical", command=self.listbox.yview)
        scrollbar.pack(side="right", fill="y")
        self.listbox.config(yscrollcommand=scrollbar.set)

        # Controls
        controls = ttk.Frame(self.root)
        controls.pack(pady=10)

        ttk.Button(controls, text="Add Files", command=self._add_files).grid(row=0, column=0, padx=5)
        ttk.Button(controls, text="Move Up", command=self._move_up).grid(row=0, column=1, padx=5)
        ttk.Button(controls, text="Move Down", command=self._move_down).grid(row=0, column=2, padx=5)
        ttk.Button(controls, text="Remove", command=self._remove_file).grid(row=0, column=3, padx=5)
        ttk.Button(controls, text="Clear", command=self._clear_all).grid(row=0, column=4, padx=5)

        # Progress
        self.progress = ttk.Progressbar(self.root, orient="horizontal", length=400, mode="determinate")
        self.progress.pack(pady=10)

        self.status_var = tk.StringVar(value="Ready")
        ttk.Label(self.root, textvariable=self.status_var, font=('Segoe UI', 9)).pack()

        # Merge Button
        self.btn_merge = ttk.Button(self.root, text="⚡ Merge Files", command=self._start_merge)
        self.btn_merge.pack(pady=20, ipadx=20)

    # --- UI Logic ---
    def _add_files(self):
        paths = filedialog.askopenfilenames(filetypes=[("Documents", "*.pdf *.pptx *.ppt")])
        for p in paths:
            if p not in self.files:
                self.files.append(p)
                self.listbox.insert(tk.END, Path(p).name)

    def _remove_file(self):
        selected = self.listbox.curselection()
        if selected:
            idx = selected[0]
            self.files.pop(idx)
            self.listbox.delete(idx)

    def _move_up(self):
        selected = self.listbox.curselection()
        if selected and selected[0] > 0:
            idx = selected[0]
            self.files[idx], self.files[idx-1] = self.files[idx-1], self.files[idx]
            self._refresh_listbox(idx-1)

    def _move_down(self):
        selected = self.listbox.curselection()
        if selected and selected[0] < len(self.files) - 1:
            idx = selected[0]
            self.files[idx], self.files[idx+1] = self.files[idx+1], self.files[idx]
            self._refresh_listbox(idx+1)

    def _clear_all(self):
        self.files.clear()
        self.listbox.delete(0, tk.END)

    def _refresh_listbox(self, new_index):
        self.listbox.delete(0, tk.END)
        for f in self.files:
            self.listbox.insert(tk.END, Path(f).name)
        self.listbox.select_set(new_index)

    # --- Core Processing Logic ---
    def _start_merge(self):
        if len(self.files) < 2:
            messagebox.showwarning("Warning", "Please add at least 2 files.")
            return
        self.btn_merge.state(['disabled'])
        threading.Thread(target=self._merge_worker, daemon=True).start()

    def _merge_worker(self):
        try:
            exts = {Path(f).suffix.lower() for f in self.files}
            all_pdf = exts <= {'.pdf'}
            all_pptx = exts <= {'.pptx', '.ppt'}

            if all_pdf:
                out = self._merge_pdfs(self.files)
            elif all_pptx:
                out = self._merge_pptx(self.files)
            else:
                out = self._merge_mixed(self.files)

            messagebox.showinfo("Success", f"Merged file saved to:\n{out}")
        except Exception as e:
            messagebox.showerror("Error", str(e))
        finally:
            self.progress['value'] = 0
            self.status_var.set("Ready")
            self.btn_merge.state(['!disabled'])

    def _merge_pdfs(self, paths):
        from pypdf import PdfWriter, PdfReader
        writer = PdfWriter()
        for i, p in enumerate(paths):
            self.status_var.set(f"Reading {Path(p).name}...")
            self.progress['value'] = (i / len(paths)) * 100
            reader = PdfReader(p)
            for page in reader.pages:
                writer.add_page(page)
        
        out = self._get_out_path("merged.pdf")
        with open(out, "wb") as f:
            writer.write(f)
        return out

    def _merge_pptx(self, paths):
        from pptx import Presentation
        import copy
        base = Presentation(paths[0])
        for i, p in enumerate(paths[1:], 1):
            self.status_var.set(f"Merging {Path(p).name}...")
            self.progress['value'] = (i / len(paths)) * 100
            src = Presentation(p)
            for slide in src.slides:
                layout_idx = min(src.slides.index(slide), len(base.slide_layouts)-1)
                new_slide = base.slides.add_slide(base.slide_layouts[layout_idx])
                for ph in new_slide.placeholders:
                    sp = ph._element
                    sp.getparent().remove(sp)
                for shape in slide.shapes:
                    el = copy.deepcopy(shape._element)
                    new_slide.shapes._spTree.insert(2, el)
        out = self._get_out_path("merged.pptx")
        base.save(out)
        return out

    def _merge_mixed(self, paths):
        tmpdir = tempfile.mkdtemp()
        pdf_paths = []
        try:
            for i, p in enumerate(paths):
                self.status_var.set(f"Processing {Path(p).name}...")
                self.progress['value'] = (i / len(paths)) * 100
                if Path(p).suffix.lower() == '.pdf':
                    pdf_paths.append(p)
                else:
                    pdf_paths.append(self._pptx_to_pdf(p, tmpdir))
            return self._merge_pdfs(pdf_paths)
        finally:
            shutil.rmtree(tmpdir, ignore_errors=True)

    def _pptx_to_pdf(self, pptx_path, out_dir):
        subprocess.run(["libreoffice", "--headless", "--convert-to", "pdf", "--outdir", out_dir, pptx_path], check=True)
        return str(Path(out_dir) / f"{Path(pptx_path).stem}.pdf")

    def _get_out_path(self, name):
        p = Path.home() / "Desktop" / name
        return str(p)

if __name__ == "__main__":
    root = tk.Tk()
    app = PDFPPTMerger(root)
    root.mainloop()